# Lab Work - 9.6

In [ ]:
# ============================================================
# Imports & global settings
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

from sklearn.datasets import load_iris, load_breast_cancer, fetch_20newsgroups, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, Binarizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    brier_score_loss, ConfusionMatrixDisplay
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import DecisionBoundaryDisplay

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print('Libraries loaded successfully.')

---
# Q.1 – Gaussian Naive Bayes (Iris Dataset)

## Q.1.01 – Load data, fit GaussianNB (raw vs scaled), compare accuracy

In [ ]:
# Load Iris and split 80/20 stratified
iris = load_iris()
X, y = iris.data, iris.target
feature_names = iris.feature_names
target_names = iris.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Class distribution (train): {np.bincount(y_train)}')
print(f'Class distribution (test):  {np.bincount(y_test)}')

In [ ]:
# --- Raw (unscaled) GaussianNB ---
gnb_raw = GaussianNB()
gnb_raw.fit(X_train, y_train)
y_pred_raw = gnb_raw.predict(X_test)
acc_raw = accuracy_score(y_test, y_pred_raw)

print('=== GaussianNB on RAW (unscaled) data ===')
print(f'Test Accuracy: {acc_raw:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_raw, target_names=target_names))

In [ ]:
# --- Scaled GaussianNB (StandardScaler fit on train only) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

gnb_scaled = GaussianNB()
gnb_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = gnb_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, y_pred_scaled)

print('=== GaussianNB on SCALED data ===')
print(f'Test Accuracy: {acc_scaled:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_scaled, target_names=target_names))

print('\n--- Accuracy Comparison ---')
print(f'Raw accuracy:    {acc_raw:.4f}')
print(f'Scaled accuracy: {acc_scaled:.4f}')

### Why is Gaussian NB less sensitive to scaling than SVM or KNN?

Gaussian Naive Bayes models **each feature’s distribution independently** as a univariate Gaussian:

$$
P(x_j \mid y=c) = \frac{1}{\sqrt{2\pi\sigma_{c,j}^2}}\exp\left(-\frac{(x_j-\mu_{c,j})^2}{2\sigma_{c,j}^2}\right)
$$

- The mean $\mu$ and variance $\sigma^2$ are estimated **per feature, per class**.
- Scaling a feature simply rescales its mean and variance by the same factor; the standardized distance $(x-\mu)/\sigma$ remains unchanged.
- Consequently the likelihood values (and therefore the posterior probabilities) are invariant to linear scaling of any individual feature.

In contrast:
- **KNN** uses Euclidean (or other) distance that mixes all features; a feature with larger numeric range dominates the distance.
- **SVM** (with RBF or linear kernels) also relies on distances / margins that are scale-dependent unless the data are standardized.

Hence Gaussian NB is essentially scale-invariant (for each feature independently), while distance-based methods are not.

## Q.1.02 – Inspect fitted parameters $\theta$ (means) and $\sigma^2$ (variances)

In [ ]:
print('model.theta_ (mean per feature per class) shape:', gnb_raw.theta_.shape)
print('(n_classes, n_features) =', gnb_raw.theta_.shape)
print('\nMeans (theta_):')
print(pd.DataFrame(gnb_raw.theta_, index=target_names, columns=feature_names).round(4))

print('\nmodel.var_ (variance per feature per class) shape:', gnb_raw.var_.shape)
print('\nVariances (var_):')
print(pd.DataFrame(gnb_raw.var_, index=target_names, columns=feature_names).round(4))

In [ ]:
# Verify that theta_[class_idx, feature_idx] equals the sample mean of that feature for that class
print('Verification of sample means vs model.theta_:')
for c in range(3):
    mask = (y_train == c)
    for f in range(4):
        sample_mean = X_train[mask, f].mean()
        model_mean  = gnb_raw.theta_[c, f]
        match = np.isclose(sample_mean, model_mean)
        print(f'  Class {target_names[c]:10s} | Feature {feature_names[f]:20s} | '
              f'sample={sample_mean:.6f}  model={model_mean:.6f}  match={match}')

## Q.1.03 – Manual Gaussian likelihood for first test point, class 0 (setosa), feature 0

In [ ]:
# First test point, class 0 (setosa), feature 0 (sepal length)
x0 = X_test[0, 0]          # sepal length of first test sample
mu  = gnb_raw.theta_[0, 0]  # mean of sepal length for setosa
var = gnb_raw.var_[0, 0]    # variance of sepal length for setosa

# Manual Gaussian PDF
pdf_manual = (1.0 / np.sqrt(2 * np.pi * var)) * np.exp( -0.5 * (x0 - mu)**2 / var )

print(f'x0 (sepal length) = {x0:.4f}')
print(f'mu  (setosa)      = {mu:.4f}')
print(f'var (setosa)      = {var:.4f}')
print(f'\nManual P(x0 | class=0) = {pdf_manual:.6e}')

# Compare with sklearn's predict_proba (joint likelihood contribution is harder to extract
# directly, but we can verify the full posterior)
proba = gnb_raw.predict_proba(X_test[:1])
print(f'\nsklearn predict_proba for first test point: {proba[0]}')
print(f'  → P(class=0 | x) = {proba[0, 0]:.6f}')

In [ ]:
# Full manual log-likelihood for verification (all 4 features)
def gaussian_logpdf(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - 0.5 * (x - mean)**2 / var

log_lik = 0.0
for f in range(4):
    log_lik += gaussian_logpdf(X_test[0, f], gnb_raw.theta_[0, f], gnb_raw.var_[0, f])

log_prior = np.log(gnb_raw.class_prior_[0])
log_post  = log_lik + log_prior

print(f'Manual log-likelihood (class 0): {log_lik:.6f}')
print(f'Manual log-prior:                {log_prior:.6f}')
print(f'Manual un-normalized log-post:   {log_post:.6f}')

## Q.1.04 – Plot class-conditional Gaussian distributions (4 subplots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for f_idx, ax in enumerate(axes):
    feat = feature_names[f_idx]
    # data range for this feature
    xmin, xmax = X[:, f_idx].min() - 0.5, X[:, f_idx].max() + 0.5
    x_grid = np.linspace(xmin, xmax, 300)

    for c in range(3):
        mu  = gnb_raw.theta_[c, f_idx]
        std = np.sqrt(gnb_raw.var_[c, f_idx])
        pdf = norm.pdf(x_grid, mu, std)

        ax.plot(x_grid, pdf, color=colors[c], lw=2, label=f'{target_names[c]}')
        # ±1σ shaded region
        ax.fill_between(x_grid, 0, pdf,
                        where=(x_grid >= mu - std) & (x_grid <= mu + std),
                        color=colors[c], alpha=0.25)

    ax.set_title(f'Feature: {feat}', fontsize=12)
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Class-conditional Gaussian distributions learned by GaussianNB\n'
             '(solid curves = PDF, shaded = mean ± 1σ)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Interpretation: Overlapping regions of the Gaussians are exactly where misclassification is most likely.')

## Q.1.05 – Compare GaussianNB vs LogisticRegression vs KNN (k=5)

In [ ]:
models = {
    'GaussianNB': GaussianNB(),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'KNeighbors (k=5)': KNeighborsClassifier(n_neighbors=5)
}

results = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    # One-vs-rest AUC-ROC
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

    results.append({'Model': name, 'Accuracy': acc, 'F1-weighted': f1, 'AUC-ROC (ovr)': auc})
    print(f'{name:20s}  Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')

pd.DataFrame(results).set_index('Model')

### Why is Gaussian NB competitive on Iris despite the independence assumption?

Iris has only **4 features** that are only moderately correlated within each class.  
The independence assumption is therefore only mildly violated.  
In addition:
- Class-conditional distributions are reasonably close to Gaussian.
- The decision boundaries of Gaussian NB turn out to be close to those of the more flexible models on this simple data set.
- With tiny sample size (120 training points) the strong inductive bias of NB acts as a regularizer, preventing over-fitting that a more complex model might suffer.

## Q.1.06 – Calibration curve (reliability diagram) for GaussianNB

In [ ]:
# Use one-vs-rest: probability of the true class
y_proba_gnb = gnb_raw.predict_proba(X_test)
# For multi-class we look at the predicted probability of the positive class in a binary sense,
# but a simple reliability diagram can be drawn for each class or for the max probability.
# Here we plot for class 0 (setosa) as an example, and also the overall max-prob calibration.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Reliability diagram for each class (one-vs-rest) ---
for c in range(3):
    y_bin = (y_test == c).astype(int)
    prob_pos = y_proba_gnb[:, c]
    frac_pos, mean_pred = calibration_curve(y_bin, prob_pos, n_bins=5, strategy='uniform')
    axes[0].plot(mean_pred, frac_pos, 's-', label=f'{target_names[c]}')

axes[0].plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of positives')
axes[0].set_title('Reliability diagram (one-vs-rest)')
axes[0].legend()
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

# --- Isotonic calibration ---
calibrated = CalibratedClassifierCV(GaussianNB(), method='isotonic', cv=3)
calibrated.fit(X_train, y_train)
y_proba_cal = calibrated.predict_proba(X_test)

for c in range(3):
    y_bin = (y_test == c).astype(int)
    prob_pos = y_proba_cal[:, c]
    frac_pos, mean_pred = calibration_curve(y_bin, prob_pos, n_bins=5, strategy='uniform')
    axes[1].plot(mean_pred, frac_pos, 's-', label=f'{target_names[c]}')

axes[1].plot([0, 1], [0, 1], 'k--', label='Perfectly calibrated')
axes[1].set_xlabel('Mean predicted probability')
axes[1].set_ylabel('Fraction of positives')
axes[1].set_title('After Isotonic Calibration')
axes[1].legend()
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print('GaussianNB on Iris is already reasonably well-calibrated because the class-conditional densities are simple and the data are well-separated for setosa.  Calibration can still improve the other two classes slightly.')

---
# Q.2 – Multinomial & Bernoulli Naive Bayes (20 Newsgroups)

## Q.2.01 – Load 20 Newsgroups subset, CountVectorizer, document-term matrix

In [ ]:
categories = ['rec.sport.hockey', 'sci.med', 'talk.politics.guns']

newsgroups_train = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)
newsgroups_test = fetch_20newsgroups(
    subset='test',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print(f'Train documents: {len(newsgroups_train.data)}')
print(f'Test documents:  {len(newsgroups_test.data)}')
print(f'Target names:    {newsgroups_train.target_names}')
print(f'Class counts (train): {np.bincount(newsgroups_train.target)}')

In [ ]:
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
X_train_counts = vectorizer.fit_transform(newsgroups_train.data)
X_test_counts  = vectorizer.transform(newsgroups_test.data)

print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print(f'Document-term matrix shape (train): {X_train_counts.shape}')
print(f'Document-term matrix shape (test):  {X_test_counts.shape}')
print(f'Sparsity: {1 - X_train_counts.nnz / (X_train_counts.shape[0]*X_train_counts.shape[1]):.4f}')

## Q.2.02 – Fit MultinomialNB, accuracy, F1, confusion matrix

In [ ]:
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_train_counts, newsgroups_train.target)
y_pred_mnb = mnb.predict(X_test_counts)

acc_mnb = accuracy_score(newsgroups_test.target, y_pred_mnb)
f1_mnb  = f1_score(newsgroups_test.target, y_pred_mnb, average='weighted')

print(f'MultinomialNB  Accuracy = {acc_mnb:.4f}')
print(f'MultinomialNB  F1-weighted = {f1_mnb:.4f}')
print('\nClassification Report:')
print(classification_report(newsgroups_test.target, y_pred_mnb,
                            target_names=newsgroups_train.target_names))

cm = confusion_matrix(newsgroups_test.target, y_pred_mnb)
print('\nConfusion Matrix:')
print(cm)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=newsgroups_train.target_names,
            yticklabels=newsgroups_train.target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('MultinomialNB Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Identify most confused categories
print('Most confused pair (off-diagonal maximum):')
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
i, j = np.unravel_index(cm_off.argmax(), cm_off.shape)
print(f'  True {newsgroups_train.target_names[i]}  →  Predicted {newsgroups_train.target_names[j]}  ({cm[i,j]} times)')

print('\nExplanation: Categories that share many vocabulary tokens (e.g. political / medical terminology) '
      'tend to be confused. Look at the top overlapping words between the two classes.')

## Q.2.03 – BernoulliNB on binary presence/absence matrix

In [ ]:
# Option 1: binarize=0.0 inside BernoulliNB
bnb = BernoulliNB(alpha=1.0, binarize=0.0)
bnb.fit(X_train_counts, newsgroups_train.target)
y_pred_bnb = bnb.predict(X_test_counts)

acc_bnb = accuracy_score(newsgroups_test.target, y_pred_bnb)
f1_bnb  = f1_score(newsgroups_test.target, y_pred_bnb, average='weighted')

print(f'BernoulliNB   Accuracy = {acc_bnb:.4f}')
print(f'BernoulliNB   F1-weighted = {f1_bnb:.4f}')
print('\nClassification Report:')
print(classification_report(newsgroups_test.target, y_pred_bnb,
                            target_names=newsgroups_train.target_names))

print('\n--- Comparison ---')
print(f'MultinomialNB F1 = {f1_mnb:.4f}')
print(f'BernoulliNB   F1 = {f1_bnb:.4f}')

### Why does one outperform the other?

- **MultinomialNB** models **term frequency** (how many times a word appears).  
  Documents in 20 Newsgroups vary greatly in length; raw counts therefore carry useful signal.
- **BernoulliNB** only models **presence / absence**.  All frequency information is discarded.

On this data set the length / frequency distribution is informative, so MultinomialNB usually wins.  
BernoulliNB can be competitive (or better) when documents are very short (e.g. tweets) where presence is more important than count.

## Q.2.04 – Top-15 most informative words per class (feature_log_prob_)

In [ ]:
feature_names_cv = np.array(vectorizer.get_feature_names_out())

print('Top-15 most informative words per class (highest feature_log_prob_):\n')
for c, class_name in enumerate(newsgroups_train.target_names):
    # sort indices of log-prob in descending order
    top_idx = np.argsort(mnb.feature_log_prob_[c])[::-1][:15]
    top_words = feature_names_cv[top_idx]
    top_logp  = mnb.feature_log_prob_[c][top_idx]
    print(f'=== {class_name} ===')
    for w, lp in zip(top_words, top_logp):
        print(f'  {w:20s}  logP = {lp:.4f}')
    print()

## Q.2.05 – TF-IDF instead of raw counts

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(newsgroups_train.data)
X_test_tfidf  = tfidf.transform(newsgroups_test.data)

mnb_tfidf = MultinomialNB(alpha=1.0)
mnb_tfidf.fit(X_train_tfidf, newsgroups_train.target)
y_pred_tfidf = mnb_tfidf.predict(X_test_tfidf)

acc_tfidf = accuracy_score(newsgroups_test.target, y_pred_tfidf)
f1_tfidf  = f1_score(newsgroups_test.target, y_pred_tfidf, average='weighted')

print(f'TF-IDF MultinomialNB  Accuracy = {acc_tfidf:.4f}')
print(f'TF-IDF MultinomialNB  F1-weighted = {f1_tfidf:.4f}')
print(f'\nRaw-count MultinomialNB F1 = {f1_mnb:.4f}')
print(f'TF-IDF MultinomialNB    F1 = {f1_tfidf:.4f}')

print('\nExplanation: TF-IDF down-weights common words that appear in many documents. '
      'This often improves discrimination, especially when stop-words are already removed. '
      'On this particular 3-class subset the gain may be modest.')

## Q.2.06 – Full Pipeline + GridSearchCV

In [ ]:
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', MultinomialNB())
])

param_grid = {
    'tfidf__max_features': [2000, 5000, 10000],
    'clf__alpha': [0.01, 0.1, 0.5, 1.0, 5.0]
}

gs = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)
gs.fit(newsgroups_train.data, newsgroups_train.target)

print('Best parameters:', gs.best_params_)
print(f'Best CV F1-weighted: {gs.best_score_:.4f}')

y_pred_gs = gs.predict(newsgroups_test.data)
test_f1 = f1_score(newsgroups_test.target, y_pred_gs, average='weighted')
print(f'Test F1-weighted:    {test_f1:.4f}')

---
# Q.3 – Tuning, Calibration & Multi-class Evaluation

## Q.3.01 – Tune `var_smoothing` for GaussianNB on Breast Cancer

In [ ]:
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=42
)

var_smoothing_values = [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]
cv_scores = []

print(f'{"var_smoothing":>15s}  {"Mean 5-fold F1":>15s}')
print('-' * 35)
for vs in var_smoothing_values:
    gnb = GaussianNB(var_smoothing=vs)
    scores = cross_val_score(gnb, X_bc_train, y_bc_train,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='f1_weighted')
    mean_f1 = scores.mean()
    cv_scores.append(mean_f1)
    print(f'{vs:15.0e}  {mean_f1:15.4f}')

best_vs = var_smoothing_values[np.argmax(cv_scores)]
print(f'\nBest var_smoothing = {best_vs}')

### What does `var_smoothing` do?

`var_smoothing` adds a fraction of the **largest variance across all features** to every feature’s variance:

$$
\sigma_{c,j}^{2} \leftarrow \sigma_{c,j}^{2} + \varepsilon \cdot \max_{j'}\sigma_{c,j'}^{2}
$$

This prevents zero-variance (or near-zero-variance) features from producing infinite / extremely large likelihoods and stabilises the Gaussian density estimates.

## Q.3.02 – Best GaussianNB vs default; Brier score

In [ ]:
# Best model
gnb_best = GaussianNB(var_smoothing=best_vs)
gnb_best.fit(X_bc_train, y_bc_train)
y_pred_best = gnb_best.predict(X_bc_test)
y_proba_best = gnb_best.predict_proba(X_bc_test)[:, 1]

# Default model
gnb_def = GaussianNB(var_smoothing=1e-9)
gnb_def.fit(X_bc_train, y_bc_train)
y_pred_def = gnb_def.predict(X_bc_test)
y_proba_def = gnb_def.predict_proba(X_bc_test)[:, 1]

def report(name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='weighted')
    auc = roc_auc_score(y_true, y_proba)
    brier = brier_score_loss(y_true, y_proba)
    print(f'{name:25s}  Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}  Brier={brier:.4f}')

print(f'{"Model":25s}  {"Acc":>8s}  {"F1":>8s}  {"AUC":>8s}  {"Brier":>8s}')
report('GaussianNB (best vs)', y_bc_test, y_pred_best, y_proba_best)
report('GaussianNB (default)', y_bc_test, y_pred_def,  y_proba_def)

print('\nBrier score = (1/n) Σ (p_i − y_i)²   →  lower is better (better calibrated probabilities).')

## Q.3.03 – ROC & Precision-Recall curves (Breast Cancer)

In [ ]:
# Logistic Regression for comparison
lr = LogisticRegression(max_iter=5000, random_state=42)
lr.fit(X_bc_train, y_bc_train)
y_proba_lr = lr.predict_proba(X_bc_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
for name, proba in [('GaussianNB (best)', y_proba_best), ('LogisticRegression', y_proba_lr)]:
    fpr, tpr, _ = roc_curve(y_bc_test, proba)
    auc = roc_auc_score(y_bc_test, proba)
    axes[0].plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve – Breast Cancer')
axes[0].legend(loc='lower right')

# Precision-Recall
for name, proba in [('GaussianNB (best)', y_proba_best), ('LogisticRegression', y_proba_lr)]:
    prec, rec, _ = precision_recall_curve(y_bc_test, proba)
    ap = average_precision_score(y_bc_test, proba)
    axes[1].plot(rec, prec, lw=2, label=f'{name} (AP={ap:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve – Breast Cancer')
axes[1].legend(loc='lower left')

plt.tight_layout()
plt.show()

pos_rate = y_bc_test.mean()
print(f'Positive class rate in test set ≈ {pos_rate:.1%}')
print('\nWhy PR curve is more informative for imbalanced data:')
print('ROC treats true-negatives and false-positives symmetrically.  When the negative class dominates,')
print('a large number of true negatives can make the ROC look good even if precision is poor.')
print('PR curve focuses on the positive class and therefore better reflects performance under imbalance.')

## Q.3.04 – Full confusion-matrix heatmap on Iris + per-class metrics

In [ ]:
y_pred_iris = gnb_raw.predict(X_test)
cm_iris = confusion_matrix(y_test, y_pred_iris)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=target_names, yticklabels=target_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('GaussianNB – Iris Confusion Matrix')
plt.tight_layout()
plt.show()

# Per-class precision, recall, F1
report_dict = classification_report(y_test, y_pred_iris, target_names=target_names, output_dict=True)
print(pd.DataFrame(report_dict).T.round(4))

print('\nClass with lowest recall usually corresponds to the class whose feature distributions')
print('overlap most with the other classes (visually visible in the density plots of Q.1.04).')

## Q.3.05 – Naive Bayes from scratch (binary classification)

In [ ]:
class GaussianNB_FromScratch:
    """Minimal Gaussian Naive Bayes for binary / multi-class classification."""
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        self.class_prior_ = np.zeros(n_classes)
        self.theta_ = np.zeros((n_classes, n_features))   # means
        self.var_   = np.zeros((n_classes, n_features))   # variances

        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            self.class_prior_[i] = len(X_c) / len(y)
            self.theta_[i] = X_c.mean(axis=0)
            self.var_[i]   = X_c.var(axis=0) + 1e-9   # small smoothing
        return self

    def _log_likelihood(self, X):
        # (n_samples, n_classes)
        n_samples = X.shape[0]
        n_classes = len(self.classes_)
        log_lik = np.zeros((n_samples, n_classes))
        for i in range(n_classes):
            # sum over features of log N(x_j; mu, var)
            log_lik[:, i] = np.sum(
                -0.5 * np.log(2 * np.pi * self.var_[i])
                -0.5 * (X - self.theta_[i])**2 / self.var_[i],
                axis=1
            )
        return log_lik

    def predict_proba(self, X):
        log_prior = np.log(self.class_prior_)
        log_post  = self._log_likelihood(X) + log_prior
        # numerical stability
        log_post -= log_post.max(axis=1, keepdims=True)
        post = np.exp(log_post)
        post /= post.sum(axis=1, keepdims=True)
        return post

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# Test on Breast Cancer
scratch = GaussianNB_FromScratch()
scratch.fit(X_bc_train, y_bc_train)
y_pred_scratch = scratch.predict(X_bc_test)
acc_scratch = accuracy_score(y_bc_test, y_pred_scratch)

gnb_ref = GaussianNB()
gnb_ref.fit(X_bc_train, y_bc_train)
acc_ref = accuracy_score(y_bc_test, gnb_ref.predict(X_bc_test))

print(f'From-scratch GaussianNB accuracy: {acc_scratch:.4f}')
print(f'sklearn GaussianNB accuracy:      {acc_ref:.4f}')
print(f'Match: {np.isclose(acc_scratch, acc_ref, atol=0.01)}')

## Q.3.06 – Effect of feature correlation on GaussianNB vs Logistic Regression

In [ ]:
# Dataset 1: truly independent features
X_ind, y_ind = make_classification(
    n_samples=1000, n_features=5, n_informative=5, n_redundant=0,
    n_clusters_per_class=1, random_state=42
)
X_ind_tr, X_ind_te, y_ind_tr, y_ind_te = train_test_split(
    X_ind, y_ind, test_size=0.3, random_state=42
)

# Dataset 2: high correlation (many redundant features)
X_cor, y_cor = make_classification(
    n_samples=1000, n_features=15, n_informative=5, n_redundant=10,
    n_clusters_per_class=1, random_state=42
)
X_cor_tr, X_cor_te, y_cor_tr, y_cor_te = train_test_split(
    X_cor, y_cor, test_size=0.3, random_state=42
)

def evaluate(Xtr, Xte, ytr, yte, name):
    gnb = GaussianNB().fit(Xtr, ytr)
    lr  = LogisticRegression(max_iter=2000, random_state=42).fit(Xtr, ytr)
    print(f'{name:30s}  GNB Acc={accuracy_score(yte, gnb.predict(Xte)):.4f}  '
          f'LR Acc={accuracy_score(yte, lr.predict(Xte)):.4f}')

print('Independent features (n_informative=5, n_redundant=0):')
evaluate(X_ind_tr, X_ind_te, y_ind_tr, y_ind_te, 'Independent')

print('\nHighly correlated features (n_informative=5, n_redundant=10):')
evaluate(X_cor_tr, X_cor_te, y_cor_tr, y_cor_te, 'Correlated')

print('\nExplanation: GaussianNB multiplies the per-feature likelihoods, counting correlated')
print('features multiple times → over-confident / biased posteriors.  Logistic Regression')
print('estimates a joint weight vector and can down-weight redundant features, so it is far')
print('more robust to correlation.')

---
# Q.4 – Deep Intuition

## Q.4.01 – Performance gap when features are correlated

**Observation**  
GaussianNB ≈ 93 % on Iris (low correlation) but only ≈ 72 % on a data set with 15 highly correlated features.  
Logistic Regression stays strong (88 % → 91 %).

**Why?**

- Naive Bayes assumes **conditional independence**.  When features are correlated the joint likelihood is the product of the marginals, which **over-counts** the evidence.  The resulting posterior becomes over-confident and the decision boundary is distorted.
- Logistic Regression models $P(y\mid x)$ directly and estimates a single weight vector.  Correlated features simply share the “explanatory power”; the optimiser can shrink the redundant weights.  No independence assumption is required.

Hence the performance gap widens exactly when the independence assumption is strongly violated.

## Q.4.02 – Class-imbalance problem (class 2 always predicted as class 0)

**Diagnosis**  
Training sizes: class 0 = 10 000, class 1 = 8 000, class 2 = 200.  
The empirical class prior $P(y=2)$ is tiny (≈ 1 %).  Even if the likelihood $P(x\mid y=2)$ is higher for a true class-2 document, the prior term dominates and the posterior for class 2 never wins.

**Two corrective strategies**

1. **Data-level**  
   - Oversample the minority class (SMOTE, random oversampling) or undersample the majority classes so that the training set becomes roughly balanced.

2. **Model-level**  
   - Supply a custom `class_prior` (or use `fit_prior=False` together with a manually chosen prior) that gives class 2 a higher weight, e.g.
   ```python
   MultinomialNB(class_prior=[0.4, 0.4, 0.2])   # or any values that sum to 1
   ```
   - Alternatively use `ComplementNB` which is designed to be more robust to imbalance.

## Q.4.03 – Comparison table of NB variants

In [ ]:
comparison = pd.DataFrame({
    'Variant': ['GaussianNB', 'MultinomialNB', 'BernoulliNB', 'ComplementNB'],
    'Suitable feature type': [
        'Continuous / real-valued',
        'Non-negative counts / frequencies',
        'Binary (0/1) presence/absence',
        'Counts (especially imbalanced text)'
    ],
    'Parameters estimated': [
        'mean & variance per feature per class',
        'smoothed relative frequency of each term',
        'probability that a feature is present',
        'weights derived from complement class frequencies'
    ],
    'Ideal use case': [
        'Sensor data, medical measurements, Iris-like data',
        'Document classification with term counts / TF-IDF',
        'Short text, bag-of-words presence only',
        'Text classification with severe class imbalance'
    ],
    'Sensitivity to class imbalance': [
        'Moderate (prior can be adjusted)',
        'High (prior dominates for rare classes)',
        'High',
        'Low (designed to mitigate imbalance)'
    ]
}).set_index('Variant')

pd.set_option('display.max_colwidth', 60)
display(comparison)

print('\nFor a tweet-sentiment classifier with 500 k samples:')
print('→ Prefer MultinomialNB (or ComplementNB if the sentiment classes are heavily skewed).')
print('  Tweets are short; presence + frequency both matter, and MultinomialNB scales excellently.')

## Q.4.04 – Generative → Discriminative spectrum

$$
\underbrace{\text{Naive Bayes}}_{\text{fully generative}}
\;\rightarrow\;
\underbrace{\text{LDA}}_{\text{generative but with linear decision boundary}}
\;\rightarrow\;
\underbrace{\text{Logistic Regression}}_{\text{fully discriminative}}
$$

| Aspect | Generative (NB) | Discriminative (LR) |
|--------|-----------------|---------------------|
| Models | $P(x\mid y)$ and $P(y)$ then uses Bayes | $P(y\mid x)$ directly |
| Independence assumption | Strong (features independent given class) | None |
| Sample efficiency | High – can learn with very few examples | Needs more data to estimate the decision boundary |
| Robustness to misspecification | Poor when independence is violated | Better – only cares about the boundary |
| Calibration | Often over-confident | Usually better calibrated |

**When does Naive Bayes outperform Logistic Regression?**

- **Small $n$** (few training samples) – the strong bias of NB acts as a regulariser.
- **High $d$** relative to $n$ (text data with thousands of sparse features) – NB’s parameter count grows only linearly with $d$, while a full covariance or a dense LR can over-fit.
- When the **class-conditional independence** assumption is approximately true.

In the large-$n$, low-$d$ regime Logistic Regression (or more flexible discriminatives) usually wins.